In [ ]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm
from copy import deepcopy

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cuda", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        if damping == "zero":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 1.261),
                        kwargs.get("s18", 1.703),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": kwargs.get("rs18", 1.0),
                "alp": kwargs.get("alp", 14.0),
            }
        elif damping == "bj":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 0.3981),
                        kwargs.get("s18", 1.9889),
                        kwargs.get("rs18", 4.4211),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": self.param_vector[2],
                "alp": kwargs.get("alp", 14.0),
            }
        self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, batch_dicts):
        self.calc.reset()

        # Calculate the energy using the DFTD3 calculator
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal

    def obtain_batch_dicts(self, atoms_list):
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        return batch_dicts


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1513512_gmtkn-cc-pVDZ.csv"
)
data_name_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
data_cc_ene = data["cc_ene"].to_numpy() * 627.5094733748099
data_dft_ene = data["scf_ene"].to_numpy() * 627.5094733748099
batch_subset = [
    "W4_11",
    # "G21EA",
    # "G21IP",
    # "DIPCS10",
    # "PA26",
    # "SIE4x4",
    # "ALKBDE10",
    # "YBDE18",
    # "AL2X6",
    # "HEAVYSB11",
    # "NBPRC",
    # "ALK8",
    # "RC21",
    # "G2RC",
    # "BH76RC",
    # "FH51",
    # "TAUT15",
    # "DC13",
    # "MB16_43",
    # "DARC",
    # "RSE43",
    # "BSR36",
    # "CDIE20",
    # "ISO34",
    # # "ISOL24",
    # # "C60ISO",
    # "PArel",
    # "BH76",
    # "BHPERI",
    # "BHDIV10",
    # "INV24",
    # "BHROT27",
    # "PX13",
    # "WCPT18",
    # "RG18",
    # "ADIM6",
    # "S22",
    # "S66",
    # # "HEAVY28",
    # "WATER27",
    # "CARBHB12",
    # "PNICO23",
    # "HAL59",
    # "AHB21",
    # "CHB6",
    # "IL16",
    # "IDISP",
    # "ICONF",
    # "ACONF",
    # "Amino20x4",
    # "PCONF21",
    # "MCONF",
    # "SCONF",
    # # "UPU23",
    # "BUT14DIOL",
]

with open(f"new_dataset/gmtkn-cc-pVDZ.json") as f:
    json_data = json.load(f)

input_batch = {}
name_batch_list = {}
weight_batch_list = {}
mean_absolute_deviation = []
model = Model(device="cuda", damping="bj")
# model = Model(device="cuda", damping="zero")
model.compile(mode="max-autotune-no-cudagraphs")
for name_mol in data_name_list:
    for i_subset in batch_subset:
        if i_subset == "BH76RC":
            i_subset_name = "BH76"
        else:
            i_subset_name = i_subset
        if name_mol.startswith(i_subset_name):
            mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
            atoms = Atoms(
                symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
            )
            if i_subset not in input_batch:
                input_batch[i_subset] = []
            input_batch[i_subset].append(atoms)
            if i_subset not in name_batch_list:
                name_batch_list[i_subset] = []
            name_batch_list[i_subset].append(name_mol)

for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]
    name_batch_list[i_subset] = np.array(name_batch_list[i_subset])
    input_batch[i_subset] = model.obtain_batch_dicts(input_batch[i_subset])
    reaction_dict_copy = reaction_dict.copy()
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict_copy.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            if col.size != 1:
                print(f"Warning: {mole_name} not found in name_list")
                reaction_dict.pop(i_reaction_keys)
                break
    json_data[f"reaction-{i_subset}"] = reaction_dict

energy_batch_target = {}
for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]

    energy_batch_target[i_subset] = torch.zeros(
        len(reaction_dict), dtype=torch.float64
    )
    weight_batch = np.zeros(len(reaction_dict), dtype=np.float64)
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]
        energy_dft = 0

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            energy_dft += (data_cc_ene[col[0]] - data_dft_ene[col[0]]) * stoichiometry
            weight_batch[i_reaction_name] += data_cc_ene[col[0]] * stoichiometry
        energy_batch_target[i_subset][i_reaction_name] = energy_dft
    mean_absolute_deviation.extend(np.abs(weight_batch))
    weight_batch_list[i_subset] = 1 / np.mean(np.abs(weight_batch))

print(
    f"mean_absolute_deviation: {np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
loss_function = torch.nn.L1Loss(reduction="sum")
torch.set_printoptions(precision=5, sci_mode=False)
energy_batch_output = {}
print("start training...")

def printable(epoch):
    if epoch % 100 == 0:
        return True
    return False
if_print_step = True
parameter_list = []

for epoch in tqdm.tqdm(range(2501)):
    loss_batch = []
    wtmad_2 = 0
    optimizer.zero_grad()
    for i_subset in batch_subset:
        energy = model(input_batch[i_subset])

        reaction_dict = json_data[f"reaction-{i_subset}"]
        energy_batch_output[i_subset] = torch.zeros(
            len(reaction_dict), dtype=torch.float64
        )
        for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
            reaction_dict.items()
        ):
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]
            energy_dft = 0

            for i in range(len(systems_list)):
                mole_name = (
                    systems_list[i]
                    if i_subset == "BH76RC"
                    else f"{i_subset}-{systems_list[i]}"
                )
                stoichiometry = int(stoichiometry_list[i])

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]

                col_disp = np.where(name_batch_list[i_subset] == mole_name)[0]
                if col_disp.size == 1:
                    energy_dft += energy[col_disp[0]] * stoichiometry
                else:
                    print(f"Warning: {mole_name} not found in name_list")
                    break
            energy_batch_output[i_subset][i_reaction_name] = energy_dft
        loss = (
            loss_function(energy_batch_output[i_subset], energy_batch_target[i_subset])
            * weight_batch_list[i_subset]
        )
        loss_batch.append(
            torch.mean(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            ).item()
        )
        if printable(epoch) and if_print_step:
            parameter_dict = {}
            for key, item in model.calc.dftd_module.params.items():
                if isinstance(item, torch.Tensor):
                    parameter_dict[key] = item.detach().cpu().numpy().item()
                else:
                    parameter_dict[key] = item
            parameter_list.append(deepcopy(parameter_dict))
            print(
                f"{i_subset}, params: {parameter_dict}, mean(|E|): {torch.mean(torch.abs(energy_batch_target[i_subset])).item()}, EACH: {energy_batch_output[i_subset] - energy_batch_target[i_subset]}"
            )
        wtmad_2 += (
            torch.sum(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            )
            * weight_batch_list[i_subset]
        ).item()
        # clip the loss to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        loss.backward()
    optimizer.step()

    if printable(epoch):
        print(
            f"Epoch: {epoch}, wtmad_2: {wtmad_2 * np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}, loss: {loss_batch}"
        )

print(f"params_vector {model.calc.dftd_module.params}")

mean_absolute_deviation: 1.907452919650298
start training...


  0%|          | 1/2501 [00:00<16:15,  2.56it/s]

W4_11, params: {'s6': 1.0, 'rs6': 0.3981, 's18': 1.9889, 'rs18': 4.4211, 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.30013,      0.53794,      1.03246,      1.29847,      0.37621,
            -0.28355,      0.06165,      1.89525,     10.35723,      0.28638,
             0.24946,     -4.47341,      9.99628,      0.75359,      3.16423,
             6.01381,     -0.22225,      1.50342,      0.32446,      0.98313,
            10.89390,      3.95256,      7.26138,      5.91581,     -0.04297,
             8.25969,      5.68296,      1.94281,      5.14819,     -0.21476,
             0.54850,     -0.14887,      6.84193,     16.70592,     -0.03127,
             0.25215,      3.88717,      0.37047,      7.61492,      8.83306,
             8.15042,     34.83183,     23.20073,     12.19939,     11.90074,
            14.94296,     28.09633,     23.26021,      5.64365,      5.58089,
             3.94826,     11.97909,     13.22370,      3.35202,      4.41649,
             5.5591

  4%|▍         | 102/2501 [00:13<05:04,  7.88it/s]

W4_11, params: {'s6': 1.0, 'rs6': 0.40796933874821345, 's18': 1.9790228802052432, 'rs18': 4.430973050353992, 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.30475,      0.45491,      0.99263,      1.17222,      0.32386,
            -0.30377,      0.03091,      1.86047,      9.98086,      0.23638,
             0.18175,     -4.69149,      9.86047,      0.65805,      2.97572,
             5.67737,     -0.25324,      1.48740,      0.26848,      0.95673,
            10.69281,      3.80065,      7.18524,      5.63297,     -0.08557,
             8.01823,      5.55491,      1.79861,      5.03829,     -0.23714,
             0.52205,     -0.16058,      6.71276,     16.63779,     -0.05132,
             0.24590,      3.77161,      0.36170,      7.40903,      8.63982,
             7.93685,     34.65676,     23.12204,     12.04527,     11.71917,
            14.85646,     27.98859,     23.21251,      5.54354,      5.46384,
             3.84167,     11.93441,     12.97427,      3.2648

...

 24%|██▍       | 601/2501 [01:22<04:46,  6.64it/s]

W4_11, params: {'s6': 1.0, 'rs6': 0.4524619809119338, 's18': 1.9342370202776142, 'rs18': 4.475573018093427, 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([ -2.32240,   0.15266,   0.84927,   0.70748,   0.12847,  -0.37844,
         -0.08415,   1.73421,   8.59868,   0.04857,  -0.07337,  -5.50796,
          9.36139,   0.30604,   2.26260,   4.39782,  -0.36917,   1.42769,
          0.06225,   0.85983,   9.93270,   3.22615,   6.89903,   4.56508,
         -0.24647,   7.10333,   5.07278,   1.25833,   4.62373,  -0.31972,
          0.42242,  -0.20455,   6.22635,  16.38481,  -0.12705,   0.22226,
          3.34051,   0.32873,   6.63586,   7.91238,   7.13604,  34.01443,
         22.83016,  11.46825,  11.03803,  14.53191,  27.59735,  23.03664,
          5.17276,   5.02098,   3.44241,  11.77540,  12.03374,   2.94116,
          4.09187,   5.17377,  10.12339,   1.85773,   3.31815,   5.36902,
          3.67952,   3.52867,   9.25310,  14.53100,   9.15055,   5.83349,
         12.09998,  12.59749

 28%|██▊       | 702/2501 [01:36<04:04,  7.37it/s]

W4_11, params: {'s6': 1.0, 'rs6': 0.46058963494088556, 's18': 1.9260132105411447, 'rs18': 4.483734950913629, 'alp': 14.0}, mean(|E|): 8.912932139655672, EACH: tensor([    -2.32514,      0.10798,      0.82833,      0.63802,      0.09886,
            -0.38964,     -0.10164,      1.71561,      8.39253,      0.01993,
            -0.11240,     -5.63206,      9.28693,      0.25339,      2.15307,
             4.20025,     -0.38678,      1.41866,      0.03140,      0.84540,
             9.81606,      3.13794,      6.85534,      4.40137,     -0.27115,
             6.96259,      4.99908,      1.17614,      4.56025,     -0.33208,
             0.40719,     -0.21126,      6.15198,     16.34668,     -0.13866,
             0.21863,      3.27522,      0.32369,      6.51797,      7.80121,
             7.01408,     33.91877,     22.78622,     11.38054,     10.93428,
            14.48248,     27.53966,     23.01032,      5.11703,      4.95296,
             3.38168,     11.75245,     11.88976,      2.8924

 30%|███       | 762/2501 [01:43<03:41,  7.84it/s]

In [ ]:
data_dft_bj = []
model_new = Model(device="cuda", damping="bj", **parameter_list[10])

for name_mol in data_name_list:
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(symbols=mol.elements, positions=mol.atom_coords() * units.Bohr)
    energy = model_new(model_new.obtain_batch_dicts([atoms]))
    data_dft_bj.append(energy.item() / 627.5094733748099)

data["modified_ai_d3bj"] = data_dft_bj
data.to_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1513512_gmtkn-cc-pVDZ.csv",
    index=False,
)

TypeError: len() of unsized object

In [3]:
parameter_list[10], parameter_list[-1]

({'s6': 1.0,
  'rs6': array(0.48369235),
  's18': array(1.90256915),
  'rs18': array(4.50695005),
  'alp': 14.0},
 {'s6': 1.0,
  'rs6': array(0.58253868),
  's18': array(1.80140133),
  'rs18': array(4.6068508),
  'alp': 14.0})

In [6]:
len(parameter_list)

26